In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import matplotlib.pyplot as plt
import seaborn as sns

from warnings import filterwarnings
filterwarnings('ignore')

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/data-for-modelling/scalled-data-(01).csv
/kaggle/input/data-for-modelling/undersampled-data-(01).csv
/kaggle/input/final-data/data_bcc.csv
/kaggle/input/final-data/data_undersampled.csv


In [2]:
# metrics for model evaluation
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

metrics = []
metrics.append(recall_score)
metrics.append(f1_score)
metrics.append(accuracy_score)
metrics.append(precision_score)

# for features selection
from sklearn.feature_selection import SelectKBest

# function for doing the features selection
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import chi2
from sklearn.feature_selection import mutual_info_classif

# models initiation
models = {}

from sklearn.linear_model import LogisticRegression
models['Logistic Regression'] =  LogisticRegression()

from sklearn.svm import SVC
models['SVM'] = SVC()

from sklearn.ensemble import RandomForestClassifier
models['Random Forest'] = RandomForestClassifier()

from xgboost import XGBClassifier
models['xgboost'] = XGBClassifier()

from sklearn.svm import LinearSVC

# for finding hyperparameter
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# for splitting the dataset
from sklearn.model_selection import train_test_split

# helper function
from scipy.stats import loguniform, randint, uniform

# for doing ensemble
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier

# saving model
import pickle

In [3]:
# using the scaled data
df = pd.read_csv('/kaggle/input/final-data/data_bcc.csv')
df_undersampled = pd.read_csv('/kaggle/input/final-data/data_undersampled.csv') # with the dataset being undersampled\

In [4]:
# dropping columns
column_to_drop = ['is_student']
for col in column_to_drop:
    df.drop([col], axis=1, inplace=True)
    df_undersampled.drop([col], axis=1, inplace=True)

In [5]:
# dropping null value in the datasets
df.dropna(inplace=True)
df_undersampled.dropna(inplace=True)

In [6]:
df.shape

(27770, 19)

In [7]:
df.head()

,age,academic_pressure,cgpa,study_satisfaction,work_study_hours,financial_stress,depression,sleep_duration_encoded,suicidal_thoughts_bool,family_history_bool,sleep_duration_encoded_2,dietary_habits_encoded,nomor_degree,gender_Female,gender_Male,is_18-34,kategori_umur_18-27,kategori_umur_28-37,kategori_umur_38-59
0,0.365854,1.0,0.792757,0.4,0.250000,0.00,1.0,0.333333,1.0,0.0,0.454545,1.0,0.5,0.0,1.0,1.0,0.0,1.0,0.0
1,0.146341,0.4,0.175050,1.0,0.250000,0.25,0.0,0.333333,0.0,1.0,0.454545,0.5,0.5,1.0,0.0,1.0,1.0,0.0,0.0
2,0.317073,0.6,0.402414,1.0,0.750000,0.00,0.0,0.000000,0.0,1.0,0.000000,1.0,0.5,0.0,1.0,1.0,0.0,1.0,0.0
3,0.243902,0.6,0.112676,0.4,0.333333,1.00,1.0,0.666667,1.0,1.0,0.818182,0.5,0.5,1.0,0.0,1.0,0.0,1.0,0.0
4,0.170732,0.8,0.623742,0.6,0.083333,0.00,0.0,0.333333,1.0,0.0,0.454545,0.5,1.0,1.0,0.0,1.0,1.0,0.0,0.0


In [8]:
# # checking the correlation out of the box for the dataset without being undersampled
# plt.figure(figsize=(20, 15))
# sns.heatmap(df.corr(), annot=True)
# plt.show()

In [9]:
df_undersampled.head()

,age,academic_pressure,cgpa,study_satisfaction,work_study_hours,financial_stress,sleep_duration_encoded,suicidal_thoughts_bool,family_history_bool,sleep_duration_encoded_2,dietary_habits_encoded,nomor_degree,gender_Female,gender_Male,is_18-34,kategori_umur_18-27,kategori_umur_28-37,kategori_umur_38-59,depression
0,0.146341,0.4,0.175050,1.0,0.250000,0.25,0.333333,0.0,1.0,0.454545,0.5,0.5,1.0,0.0,1.0,1.0,0.0,0.0,0.0
1,0.317073,0.6,0.402414,1.0,0.750000,0.00,0.000000,0.0,1.0,0.000000,1.0,0.5,0.0,1.0,1.0,0.0,1.0,0.0,0.0
2,0.170732,0.8,0.623742,0.6,0.083333,0.00,0.333333,1.0,0.0,0.454545,0.5,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0
3,0.268293,0.4,0.134809,0.6,0.333333,0.00,0.000000,0.0,0.0,0.000000,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0
4,0.292683,0.6,0.907445,0.8,0.083333,0.25,0.666667,0.0,0.0,0.818182,1.0,0.5,0.0,1.0,1.0,0.0,1.0,0.0,0.0


In [10]:
# # checking the correlation out of the box for the dataset without being undersampled
# plt.figure(figsize=(20, 15))
# sns.heatmap(df_undersampled.corr(), annot=True)
# plt.show()

### Training the model without future selection

In [11]:
def evaluate(y_test, t_pred):
    print('Evaluation: ')
    for metric in metrics:
        print(str(metric).split(' ')[1], end=': ')
        print(metric(y_test, y_pred))

In [12]:
# training for dataset without being undersampled
X, y = df.drop(['depression'], axis=1), df['depression']

for model in models:
    model = models[model]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    
    print(type(model).__name__)
    print()
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # print('Evaluation: ')
    # for metric in metrics:
    #     print(str(metric).split(' ')[1], end=': ')
    #     print(metric(y_test, y_pred))
    evaluate(y_test, y_pred)

    print('\n===================================\n')

LogisticRegression

Evaluation: 
recall_score: 0.8929012345679013
f1_score: 0.8729631864815933
accuracy_score: 0.848397551314368
precision_score: 0.8538961038961039


SVC

Evaluation: 
recall_score: 0.8950617283950617
f1_score: 0.8728367193378479
accuracy_score: 0.8478574000720202
precision_score: 0.8516886930983847


RandomForestClassifier

Evaluation: 
recall_score: 0.8734567901234568
f1_score: 0.8629364232352492
accuracy_score: 0.8381346777097587
precision_score: 0.8526664658029527


XGBClassifier

Evaluation: 
recall_score: 0.8737654320987654
f1_score: 0.8614027080480756
accuracy_score: 0.8359740727403673
precision_score: 0.8493849384938494




In [13]:
# training with undersampled dataset
X, y = df_undersampled.drop(['depression'], axis=1), df_undersampled['depression']

for model in models:
    model = models[model]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
    
    print(type(model).__name__)
    print()
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    # print('Evaluation: ')
    # for metric in metrics:
    #     print(str(metric).split(' ')[1], end=': ')
    #     print(metric(y_test, y_pred))
    evaluate(y_test, y_pred)

    print('\n===================================\n')

LogisticRegression

Evaluation: 
recall_score: 0.8555893866898652
f1_score: 0.8460215053763441
accuracy_score: 0.8446180555555556
precision_score: 0.836665248830285


SVC

Evaluation: 
recall_score: 0.8499347542409743
f1_score: 0.8420599008834303
accuracy_score: 0.8409288194444444
precision_score: 0.8343296327924851


RandomForestClassifier

Evaluation: 
recall_score: 0.8464549804262723
f1_score: 0.8393357774423119
accuracy_score: 0.8383246527777778
precision_score: 0.8323353293413174


XGBClassifier

Evaluation: 
recall_score: 0.8412353197042193
f1_score: 0.8312916398022782
accuracy_score: 0.8296440972222222
precision_score: 0.8215802888700084




### Training the raw model with future selection

In [14]:
def select_n_features(df, n, display_scores=False):
    feat = list()
    X, y = df.drop(['depression'], axis=1), df['depression']
    score_functions = [f_classif, mutual_info_classif, chi2]
    for function in score_functions:
        feat_sel = SelectKBest(score_func=function, k=n)
        feat_sel.fit(X, y)
        scores = dict(zip(X.columns, feat_sel.scores_))
        sorted_scores_top_n = dict(sorted(scores.items(), key=lambda item: item[1], reverse=True)[:n])    
        print(str(function).split(' ')[1], ':')
        result = list(sorted_scores_top_n.keys())

        if display_scores:
            print(sorted_scores_top_n)
        else:
            print(result)
        
        print('=============================================')
        feat.append(result)
    return feat

n = 10 # selecting only the n top features
display_scores = True

# for the dataset without being undersampled
print('Feature selection for the dataset without being undersampled:')
feat_df = select_n_features(df, n, display_scores)

print()

# for the dataset without being undersampled
print('Feature selection for the undersampled dataset:')
feat_df_undersampled = select_n_features(df_undersampled, n, display_scores)

Feature selection for the dataset without being undersampled:
f_classif :
{'suicidal_thoughts_bool': 11862.641441955142, 'academic_pressure': 8096.775790521619, 'financial_stress': 4228.1283841051, 'age': 1511.5617702447114, 'work_study_hours': 1271.497095141893, 'dietary_habits_encoded': 1249.3359213712624, 'kategori_umur_18-27': 997.6948319243187, 'kategori_umur_28-37': 987.0550004187917, 'study_satisfaction': 806.1816266187673, 'nomor_degree': 366.7641472921395}
mutual_info_classif :
{'suicidal_thoughts_bool': 0.1554009471608977, 'academic_pressure': 0.11897775026571367, 'financial_stress': 0.06838763400184344, 'age': 0.03142770261056138, 'dietary_habits_encoded': 0.025153632613479582, 'work_study_hours': 0.02453633718824233, 'kategori_umur_18-27': 0.023368753955104182, 'kategori_umur_28-37': 0.016935282907629468, 'study_satisfaction': 0.013661035853342662, 'nomor_degree': 0.009764810393983803}
chi2 :
{'suicidal_thoughts_bool': 3053.465726176105, 'financial_stress': 885.504893023126

In [15]:
# from collections import Counter

# Count features occurences
# count_dict = pd.Series(Counter(sum(feat_df, [])))
# count_dict_undersampled = pd.Series(Counter(sum(feat_df_undersampled, [])))

# print('without being undersampled')
# print(count_dict)
# print('being undersampled')
# print(count_dict_undersampled)

# features that are choosen
# choosen_feat = list(count_dict[:10].index)
# print('Features for the un-undersampled dataset:\n', choosen_feat)
# choosen_feat_undersampled = list(count_dict_undersampled[:10].index)
# print('Features for the undersampled dataset:\n', choosen_feat_undersampled)

In [16]:
# training for dataset without being undersampled
X, y = df.drop(['depression'], axis=1), df['depression']

for model in models:
    model = models[model]
    print(type(model).__name__)
    print()
    # trying each selected features from the three features selection methods
    for i, method in enumerate(['f_classif', 'mutual_info_classif', 'chi2']):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
        print(f'Metode feature selection: {method}')
        X_train, X_test = X_train[feat_df[i]], X_test[feat_df[i]]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        evaluate(y_test, y_pred)
        # print('Evaluation: ')
        # for metric in metrics:
        #     print(str(metric).split(' ')[1], end=': ')
        #     print(metric(y_test, y_pred))

        print()

    print('===================================\n')

LogisticRegression

Metode feature selection: f_classif
Evaluation: 
recall_score: 0.8895061728395062
f1_score: 0.8709579933514656
accuracy_score: 0.8462369463449766
precision_score: 0.8531675547661338

Metode feature selection: mutual_info_classif
Evaluation: 
recall_score: 0.8895061728395062
f1_score: 0.8709579933514656
accuracy_score: 0.8462369463449766
precision_score: 0.8531675547661338

Metode feature selection: chi2
Evaluation: 
recall_score: 0.8895061728395062
f1_score: 0.8709579933514656
accuracy_score: 0.8462369463449766
precision_score: 0.8531675547661338


SVC

Metode feature selection: f_classif
Evaluation: 
recall_score: 0.8845679012345679
f1_score: 0.8671709531013616
accuracy_score: 0.8419157364061938
precision_score: 0.8504451038575668

Metode feature selection: mutual_info_classif
Evaluation: 
recall_score: 0.8845679012345679
f1_score: 0.8671709531013616
accuracy_score: 0.8419157364061938
precision_score: 0.8504451038575668

Metode feature selection: chi2
Evaluation: 


In [17]:
# training with undersampled dataset
X, y = df_undersampled.drop(['depression'], axis=1), df_undersampled['depression']

for model in models:
    model = models[model]
    print(type(model).__name__)
    print()
    # trying each selected features from the three features selection methods
    for i, method in enumerate(['f_classif', 'mutual_info_classif', 'chi2']):
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
        print(f'Metode feature selection: {method}')
        X_train, X_test = X_train[feat_df_undersampled[i]], X_test[feat_df_undersampled[i]]
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        # print('Evaluation: ')
        # for metric in metrics:
        #     print(str(metric).split(' ')[1], end=': ')
        #     print(metric(y_test, y_pred))
        evaluate(y_test, y_pred)

        print()

    print('===================================\n')

LogisticRegression

Metode feature selection: f_classif
Evaluation: 
recall_score: 0.8555893866898652
f1_score: 0.8447498389521151
accuracy_score: 0.8430989583333334
precision_score: 0.8341815097540288

Metode feature selection: mutual_info_classif
Evaluation: 
recall_score: 0.8555893866898652
f1_score: 0.8447498389521151
accuracy_score: 0.8430989583333334
precision_score: 0.8341815097540288

Metode feature selection: chi2
Evaluation: 
recall_score: 0.8555893866898652
f1_score: 0.8447498389521151
accuracy_score: 0.8430989583333334
precision_score: 0.8341815097540288


SVC

Metode feature selection: f_classif
Evaluation: 
recall_score: 0.8534145280556764
f1_score: 0.8391787852865698
accuracy_score: 0.8368055555555556
precision_score: 0.8254101809002945

Metode feature selection: mutual_info_classif
Evaluation: 
recall_score: 0.8534145280556764
f1_score: 0.8391787852865698
accuracy_score: 0.8368055555555556
precision_score: 0.8254101809002945

Metode feature selection: chi2
Evaluation: 


### Finding hyperparameter

In [18]:
# using the dataset thats not undersampled with using the features from the chi2 method
method = 0
X, y = df.drop(['depression'], axis=1), df['depression']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_test = X_train[feat_df[method]], X_test[feat_df[method]]

In [19]:
# using the dataset thats not undersampled without using the feature selection
X, y = df.drop(['depression'], axis=1), df['depression']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

#### For the logistic regression model

##### Using the dataset with features from the chi2 method

In [20]:
model_lr = LogisticRegression()

param_grid = [
    {
        'penalty': ['l1', 'l2'],
        'C': np.logspace(-4, 4, 20),
        'solver': ['lbfgs', 'newton-cg', 'liblinear', 'sag', 'saga'],
        'max_iter': [100, 1000, 2500, 5000],
        'tol': [1e-4, 1e-3, 1e-2]
    }
]

random_search = RandomizedSearchCV(
    estimator=model_lr,
    param_distributions=param_grid,
    n_iter=100,  # Number of random combinations to try
    cv=5,        # 5-fold cross-validation
    random_state=42
)

random_search.fit(X_train, y_train)
print('Best params:', random_search.best_params_)

Best params: {'tol': 0.001, 'solver': 'liblinear', 'penalty': 'l2', 'max_iter': 5000, 'C': 1438.4498882876599}


In [21]:
y_pred = random_search.predict(X_test)
print('Evaluation: ')
for metric in metrics:
    print(str(metric).split(' ')[1], end=': ')
    print(metric(y_test, y_pred))

Evaluation: 
recall_score: 0.8922839506172839
f1_score: 0.8730182696663145
accuracy_score: 0.848577601728484
precision_score: 0.8545669524091043


##### Using the dataset with all the features

In [22]:
model_lr = LogisticRegression()

param_distributions = {
    'penalty': ['l1', 'l2', 'elasticnet', 'none'],
    'C': loguniform(1e-4, 1e4),
    'solver': ['lbfgs', 'newton-cg', 'liblinear', 'sag', 'saga'],
    'max_iter': randint(100, 5000),
    'tol': loguniform(1e-5, 1e-1)
}

random_search = RandomizedSearchCV(
    estimator=model_lr,
    param_distributions=param_distributions,
    n_iter=100,  # Number of random combinations to try
    cv=5,        # 5-fold cross-validation
    random_state=42
)

random_search.fit(X_train, y_train)

RandomizedSearchCV(cv=5, estimator=LogisticRegression(), n_iter=100,
                   param_distributions={'C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x794158a363b0>,
                                        'max_iter': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x794158c948b0>,
                                        'penalty': ['l1', 'l2', 'elasticnet',
                                                    'none'],
                                        'solver': ['lbfgs', 'newton-cg',
                                                   'liblinear', 'sag', 'saga'],
                                        'tol': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7941589c2290>},
                   random_state=42)

In [23]:
print("Best Parameters:", random_search.best_params_)

Best Parameters: {'C': 0.001963494606984601, 'max_iter': 1545, 'penalty': 'none', 'solver': 'sag', 'tol': 0.00210255074240948}


In [24]:
y_pred = random_search.predict(X_test)
print('Evaluation: ')
for metric in metrics:
    print(str(metric).split(' ')[1], end=': ')
    print(metric(y_test, y_pred))

Evaluation: 
recall_score: 0.891358024691358
f1_score: 0.8725075528700906
accuracy_score: 0.8480374504861361
precision_score: 0.8544378698224852


In [25]:
# save the iris classification model as a pickle file
model_pkl_file = "model_bcc1.pkl"  

with open(model_pkl_file, 'wb') as file:  
    pickle.dump(random_search, file)

#### For the xgboost

##### Using the dataset with features from the mutual information method

In [26]:
model_xgb = XGBClassifier()

param_dist = {
    'learning_rate': uniform(0.01, 0.3),
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 10),
    'min_child_weight': randint(1, 10),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 0.4)
}

random_search = RandomizedSearchCV(
    estimator=model_xgb,
    param_distributions=param_dist,
    n_iter=50,            # Number of parameter settings to sample
    scoring=precision_score,    # Use appropriate scoring metric
    cv=5,                 # 5-fold cross-validation
    n_jobs=-1,           # Use all available cores
    verbose=1            # Verbose output
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric=None, feature_types=None,
                                           gamma=None, grow_policy=None,
                                           importance_type=None,
                                           interaction_constraints=None,
                                           learning_rate...
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79418c272200>,
                                        'min_child_weight': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79418c272620>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x79418c270c40>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x79418c272260>},
                   scoring=<function precision_score at 0x794159c50b80>,
                   verbose=1)

In [27]:
print("Best Parameters:", random_search.best_params_)

Best Parameters: {'colsample_bytree': 0.8361397989976085, 'gamma': 0.3330780064772449, 'learning_rate': 0.022183228183253967, 'max_depth': 6, 'min_child_weight': 6, 'n_estimators': 791, 'subsample': 0.8856043945938147}


In [28]:
y_pred = random_search.predict(X_test)
print('Evaluation: ')
for metric in metrics:
    print(str(metric).split(' ')[1], end=': ')
    print(metric(y_test, y_pred))

Evaluation: 
recall_score: 0.8814814814814815
f1_score: 0.8686131386861314
accuracy_score: 0.8444364422038171
precision_score: 0.8561151079136691


##### Using the dataset with all the features

In [29]:
model_xgb = XGBClassifier()

param_dist = {
    'learning_rate': uniform(0.01, 0.3),
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 10),
    'min_child_weight': randint(1, 10),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 0.4)
}

random_search = RandomizedSearchCV(
    estimator=model_xgb,
    param_distributions=param_dist,
    n_iter=50,            # Number of parameter settings to sample
    scoring=precision_score,    # Use appropriate scoring metric
    cv=5,                 # 5-fold cross-validation
    n_jobs=-1,           # Use all available cores
    verbose=1            # Verbose output
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 50 candidates, totalling 250 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric=None, feature_types=None,
                                           gamma=None, grow_policy=None,
                                           importance_type=None,
                                           interaction_constraints=None,
                                           learning_rate...
                                        'max_depth': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x794158a10400>,
                                        'min_child_weight': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x794158a10550>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x794158a13280>,
                                        'subsample': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x794158a134f0>},
                   scoring=<function precision_score at 0x794159c50b80>,
                   verbose=1)

In [30]:
print("Best Parameters:", random_search.best_params_)

Best Parameters: {'colsample_bytree': 0.817069516563806, 'gamma': 0.09795205716698462, 'learning_rate': 0.24719312778103622, 'max_depth': 9, 'min_child_weight': 3, 'n_estimators': 695, 'subsample': 0.6204370139778311}


In [31]:
y_pred = random_search.predict(X_test)
print('Evaluation: ')
for metric in metrics:
    print(str(metric).split(' ')[1], end=': ')
    print(metric(y_test, y_pred))

Evaluation: 
recall_score: 0.8410493827160493
f1_score: 0.8356332413370132
accuracy_score: 0.806985956067699
precision_score: 0.8302864107251676


### Trying the ensemble method

In [32]:
# using the dataset thats not undersampled without using the feature selection
X, y = df.drop(['depression'], axis=1), df['depression']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

Getting the best hyperparameter for all the model

In [33]:
lr = LogisticRegression()
svc = SVC()
rf = RandomForestClassifier()
xgb = XGBClassifier()

In [34]:
# for the logistic regression model
param_distributions = {
    'penalty': ['l1', 'l2'],
    'C': loguniform(1e-4, 1e4),
    'solver': ['lbfgs', 'newton-cg', 'liblinear', 'sag', 'saga'],
    'max_iter': randint(100, 5000),
    'tol': loguniform(1e-5, 1e-1)
}

clf1 = RandomizedSearchCV(
    estimator=lr,
    param_distributions=param_distributions,
    n_iter=50,  # Number of random combinations to try
    cv=5,        # 5-fold cross-validation
    verbose=1,   # Verbose output
    scoring=precision_score,
    random_state=42
)

clf1.fit(X_train, y_train)
print("Best Parameters:", clf1.best_params_)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best Parameters: {'C': 0.09915644566638401, 'max_iter': 960, 'penalty': 'l1', 'solver': 'liblinear', 'tol': 0.013145103232150115}


In [35]:
# for the svm model
param_dist = {
    'C': uniform(1e-3, 1e3),  # Uniform distribution for C
    'kernel': ['linear', 'rbf', 'poly'],  # List of kernels
    'gamma': uniform(1e-4, 1e-1),  # Uniform distribution for gamma
    'degree': randint(1, 5)  # Integer values for degree (only for poly kernel)
}

clf2 = RandomizedSearchCV(
    estimator=svc,
    param_distributions=param_dist,
    n_iter=10,  # Number of parameter settings to sample
    cv=3,  # Cross-validation folds
    scoring=precision_score,  # Scoring metric
    random_state=42,  # For reproducibility
    verbose=1
)

clf2.fit(X_train, y_train)
print("Best Parameters:", clf2.best_params_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters: {'C': 374.54111884736244, 'degree': 1, 'gamma': 0.018443478986616378, 'kernel': 'linear'}


In [36]:
# for the random forest classifier mode
param_dist = {
    'n_estimators': randint(10, 200),  # Number of trees in the forest
    'max_depth': randint(1, 20),  # Maximum depth of the tree
    'min_samples_split': randint(2, 10),  # Minimum number of samples required to split an internal node
    'min_samples_leaf': randint(1, 10),  # Minimum number of samples required to be at a leaf node
    'max_features': ['sqrt', 'log2', None],  # Number of features to consider when looking for the best split
    'bootstrap': [True, False]  # Whether bootstrap samples are used when building trees
}

clf3 = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=15,  # Number of parameter settings to sample
    cv=5,  # Cross-validation folds
    scoring=precision_score,  # Scoring metric
    random_state=42  # For reproducibility
)

clf3.fit(X_train, y_train)
print("Best Parameters:", clf3.best_params_)

Best Parameters: {'bootstrap': True, 'max_depth': 15, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 6, 'n_estimators': 30}


In [37]:
# for the xgboost model
param_dist = {
    'learning_rate': uniform(0.01, 0.3),
    'n_estimators': randint(100, 1000),
    'max_depth': randint(3, 10),
    'min_child_weight': randint(1, 10),
    'subsample': uniform(0.5, 0.5),
    'colsample_bytree': uniform(0.5, 0.5),
    'gamma': uniform(0, 0.4)
}

clf4 = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=15,            # Number of parameter settings to sample
    scoring=precision_score,    # Use appropriate scoring metric
    cv=5,                 # 5-fold cross-validation
    n_jobs=-1,           # Use all available cores
    verbose=1            # Verbose output
)

# Fit the model
clf4.fit(X_train, y_train)
print("Best Parameters:", clf4.best_params_)

Fitting 5 folds for each of 15 candidates, totalling 75 fits
Best Parameters: {'colsample_bytree': 0.8383022731114519, 'gamma': 0.10445242460027485, 'learning_rate': 0.2326400780652121, 'max_depth': 4, 'min_child_weight': 7, 'n_estimators': 605, 'subsample': 0.6835141755613401}


In [38]:
clf1_best_params = {'C': 0.09915644566638401, 'max_iter': 960, 'penalty': 'l1', 'solver': 'liblinear', 'tol': 0.013145103232150115}
clf2_best_params = {'C': 374.54111884736244, 'degree': 1, 'gamma': 0.018443478986616378, 'kernel': 'linear'}
clf3_best_params = {'bootstrap': True, 'max_depth': 15, 'max_features': None, 'min_samples_leaf': 8, 'min_samples_split': 6, 'n_estimators': 30}
clf4_best_params = {'colsample_bytree': 0.6829070132159345, 'gamma': 0.2692258943471928, 'learning_rate': 0.021562394517052268, 'max_depth': 3, 'min_child_weight': 8, 'n_estimators': 662, 'subsample': 0.7129968204139683}

#### Voting

In [39]:
# lr = LogisticRegression(**clf1.best_params_)
# svc = SVC(**clf2.best_params_)
# rf = RandomForestClassifier(**clf3.best_params_)
# xgb = XGBClassifier(**clf4.best_params_)
lr = LogisticRegression(**clf1_best_params)
svc = SVC(**clf2_best_params)
rf = RandomForestClassifier(**clf3_best_params)
xgb = XGBClassifier(**clf4_best_params)

# using the voters with voting='hard'
vt1 = VotingClassifier(estimators=[('lr', lr), ('svc', svc), ('rf', rf), ('xgb', xgb)], voting='hard')
vt1 = vt1.fit(X_train, y_train)
y_pred = vt1.predict(X_test)
evaluate(y_test, y_pred)

Evaluation: 
recall_score: 0.8839506172839506
f1_score: 0.8711787072243347
accuracy_score: 0.8474972992437882
precision_score: 0.8587706146926537


In [40]:
# lr = LogisticRegression(**clf1.best_params_)
# svc = SVC(**clf2.best_params_, probability=True)
# rf = RandomForestClassifier(**clf3.best_params_)
# xgb = XGBClassifier(**clf4.best_params_, probability=True)
lr = LogisticRegression(**clf1_best_params)
svc = SVC(**clf2_best_params, probability=True)
rf = RandomForestClassifier(**clf3_best_params)
xgb = XGBClassifier(**clf4_best_params, probability=True)

# using the voters with voting='hard'
vt2 = VotingClassifier(estimators=
       [('lr', lr), ('svc', svc), ('rf', rf), ('xgb', xgb)],
       voting='soft', weights=[2,1,1,1],
       flatten_transform=True)
vt2 = vt2.fit(X_train, y_train)
y_pred = vt2.predict(X_test)
evaluate(y_test, y_pred)

Evaluation: 
recall_score: 0.8901234567901235
f1_score: 0.8719576719576722
accuracy_score: 0.8474972992437882
precision_score: 0.8545185185185186


#### Stacking

In [41]:
# lr = LogisticRegression(**clf1.best_params_)
# svc = SVC(**clf2.best_params_)
# rf = RandomForestClassifier(**clf3.best_params_)
# xgb = XGBClassifier(**clf4.best_params_)
lr = LogisticRegression(**clf1_best_params)
svc = SVC(**clf2_best_params)
rf = RandomForestClassifier(**clf3_best_params)
xgb = XGBClassifier(**clf4_best_params)

st = StackingClassifier(
    estimators=[('lr', lr), ('svc', svc), ('rf', rf), ('xgb', xgb)],
    n_jobs=-1,
    verbose=1
)
st.fit(X_train, y_train)
y_pred = st.predict(X_test)
evaluate(y_test, y_pred)

Evaluation: 
recall_score: 0.8944444444444445
f1_score: 0.8744719372359687
accuracy_score: 0.8501980554555275
precision_score: 0.8553719008264463


### Trying the linear svc model, knowing our dataset contains linear features

In [42]:
lcsv = LinearSVC()
lcsv.fit(X_train, y_train)
y_pred = lcsv.predict(X_test)
evaluate(y_test, y_pred)

Evaluation: 
recall_score: 0.8938271604938272
f1_score: 0.8732097090306046
accuracy_score: 0.848577601728484
precision_score: 0.8535219569702328


#### Finding the ideal hyperparameter for LinearSVC